In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('loans_full_schema.csv')

In [3]:
df['state'] = df['state'].str.upper()

In [4]:
df.head(8)

,emp_title,emp_length,state,homeownership,annual_income,verified_income,debt_to_income,annual_income_joint,verification_income_joint,debt_to_income_joint,...,sub_grade,issue_month,loan_status,initial_listing_status,disbursement_method,balance,paid_total,paid_principal,paid_interest,paid_late_fees
0,global config engineer,3.0,NJ,MORTGAGE,90000.0,Verified,18.01,NaN,NaN,NaN,...,C3,Mar-18,Current,whole,Cash,27015.86,1999.33,984.14,1015.19,0.0
1,warehouse office clerk,10.0,HI,RENT,40000.0,Not Verified,5.04,NaN,NaN,NaN,...,C1,Feb-18,Current,whole,Cash,4651.37,499.12,348.63,150.49,0.0
2,assembly,3.0,WI,RENT,40000.0,Source Verified,21.15,NaN,NaN,NaN,...,D1,Feb-18,Current,fractional,Cash,1824.63,281.80,175.37,106.43,0.0
3,customer service,1.0,PA,RENT,30000.0,Not Verified,10.16,NaN,NaN,NaN,...,A3,Jan-18,Current,whole,Cash,18853.26,3312.89,2746.74,566.15,0.0
4,security supervisor,10.0,CA,RENT,35000.0,Verified,57.96,57000.0,Verified,37.66,...,C3,Mar-18,Current,whole,Cash,21430.15,2324.65,1569.85,754.80,0.0
5,NaN,NaN,KY,OWN,34000.0,Not Verified,6.46,NaN,NaN,NaN,...,A3,Jan-18,Current,whole,Cash,4256.71,873.13,743.29,129.84,0.0
6,hr,10.0,MI,MORTGAGE,35000.0,Source Verified,23.66,155000.0,Not Verified,13.12,...,C2,Jan-18,Current,whole,Cash,22560.00,2730.51,1440.00,1290.51,0.0
7,police,10.0,AZ,MORTGAGE,110000.0,Source Verified,16.19,NaN,NaN,NaN,...,B5,Jan-18,Current,whole,Cash,19005.39,1765.84,994.61,771.23,0.0


In [7]:
df.columns

Index(['emp_title', 'emp_length', 'state', 'homeownership', 'annual_income',
       'verified_income', 'debt_to_income', 'annual_income_joint',
       'verification_income_joint', 'debt_to_income_joint', 'delinq_2y',
       'months_since_last_delinq', 'earliest_credit_line',
       'inquiries_last_12m', 'total_credit_lines', 'open_credit_lines',
       'total_credit_limit', 'total_credit_utilized',
       'num_collections_last_12m', 'num_historical_failed_to_pay',
       'months_since_90d_late', 'current_accounts_delinq',
       'total_collection_amount_ever', 'current_installment_accounts',
       'accounts_opened_24m', 'months_since_last_credit_inquiry',
       'num_satisfactory_accounts', 'num_accounts_120d_past_due',
       'num_accounts_30d_past_due', 'num_active_debit_accounts',
       'total_debit_limit', 'num_total_cc_accounts', 'num_open_cc_accounts',
       'num_cc_carrying_balance', 'num_mort_accounts',
       'account_never_delinq_percent', 'tax_liens', 'public_record_bankr

In [8]:
df.info()
# we will not delete any null values, rather we will handle them as per requirement

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 55 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   emp_title                         9167 non-null   object 
 1   emp_length                        9183 non-null   float64
 2   state                             10000 non-null  object 
 3   homeownership                     10000 non-null  object 
 4   annual_income                     10000 non-null  float64
 5   verified_income                   10000 non-null  object 
 6   debt_to_income                    9976 non-null   float64
 7   annual_income_joint               1495 non-null   float64
 8   verification_income_joint         1455 non-null   object 
 9   debt_to_income_joint              1495 non-null   float64
 10  delinq_2y                         10000 non-null  int64  
 11  months_since_last_delinq          4342 non-null   float64
 12  earli

In [5]:
df['issue_month'] = pd.to_datetime(df['issue_month'], format='%b-%y')

In [6]:
df['issue_month'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 10000 entries, 0 to 9999
Series name: issue_month
Non-Null Count  Dtype         
--------------  -----         
10000 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 78.3 KB


## DOING SOME LOGICAL CHECKS HERE

paid_total >= paid_principal + paid_interest + paid_late_fees ?

paid_principal <= loan_amount ?

balance = loan_amount - paid_principal 

In [7]:
df['correct_paid_total'] = df['paid_total'] >= df['paid_principal'] + df['paid_interest'] + df['paid_late']

KeyError: 'paid_late'

In [ ]:
df[df['correct_paid_total'] == False].count()
#1213 rows show this inconsistency

In [ ]:
df['correct_princi'] = df['paid_principal'] <= df['loan_amount']
df[df['correct_princi'] == False] # no inconsistency

In [ ]:
df['correct_balance'] = df['loan_amount'] - df['paid_principal']

In [ ]:
df[df['correct_balance'] < 0] # no inconsistency

In [ ]:
df.drop(columns =['correct_princi','correct_balance'], inplace = True) #removing above columns since needed just for checks

# ADDING SOME IMPORTANT COLUMNS

In [8]:
#calculating credit utilization ratio
df['cr_util_ratio'] = round(df['total_credit_utilized']*100.0/df['total_credit_limit'],2)

In [9]:
#calculating effective D.T.I (using joint D.T.I in joint applications)
df['effective_dti'] = df['debt_to_income']

In [10]:
df.loc[df['application_type'] == 'Joint', 'effective_dti'] = df['debt_to_income_joint']

In [11]:
df['high_debt_burden_flag'] = df['effective_dti'] >= 40.00 #widely used in consumer lending

In [12]:
df['recent_delinquency_flag'] = df['delinq_2y'] >= 1

In [13]:
df['severe_credit_event_flag'] = (df['total_collection_amount_ever'] > 0) | (df['public_record_bankrupt'] >= 1) | (df['tax_liens'] >= 1)
#these are important events, when someone goes bankrupt or someone has a tax lien(companies assume that govt should be paid first)
#or if someone's debt amount was ever sent for collections, THAT IS WHY I HAVE GROUPED THEM IN ONE

In [14]:
df['high_credit_stress_flag'] = (df['cr_util_ratio'] > 75.00) | (df['num_cc_carrying_balance'] == 3) 

In [17]:
df.head(10)

,emp_title,emp_length,state,homeownership,annual_income,verified_income,debt_to_income,annual_income_joint,verification_income_joint,debt_to_income_joint,...,paid_total,paid_principal,paid_interest,paid_late_fees,cr_util_ratio,effective_dti,high_debt_burden_flag,recent_delinquency_flag,severe_credit_event_flag,high_credit_stress_flag
0,global config engineer,3.0,NJ,MORTGAGE,90000.0,Verified,18.01,NaN,NaN,NaN,...,1999.33,984.14,1015.19,0.0,54.76,18.01,False,False,True,False
1,warehouse office clerk,10.0,HI,RENT,40000.0,Not Verified,5.04,NaN,NaN,NaN,...,499.12,348.63,150.49,0.0,15.00,5.04,False,False,True,False
2,assembly,3.0,WI,RENT,40000.0,Source Verified,21.15,NaN,NaN,NaN,...,281.80,175.37,106.43,0.0,66.13,21.15,False,False,True,False
3,customer service,1.0,PA,RENT,30000.0,Not Verified,10.16,NaN,NaN,NaN,...,3312.89,2746.74,566.15,0.0,19.67,10.16,False,False,True,False
4,security supervisor,10.0,CA,RENT,35000.0,Verified,57.96,57000.0,Verified,37.66,...,2324.65,1569.85,754.80,0.0,75.49,57.96,True,False,False,True
5,NaN,NaN,KY,OWN,34000.0,Not Verified,6.46,NaN,NaN,NaN,...,873.13,743.29,129.84,0.0,9.26,6.46,False,True,False,False
6,hr,10.0,MI,MORTGAGE,35000.0,Source Verified,23.66,155000.0,Not Verified,13.12,...,2730.51,1440.00,1290.51,0.0,6.48,23.66,False,False,False,False
7,police,10.0,AZ,MORTGAGE,110000.0,Source Verified,16.19,NaN,NaN,NaN,...,1765.84,994.61,771.23,0.0,17.76,16.19,False,True,False,False
8,parts,10.0,NV,MORTGAGE,65000.0,Source Verified,36.48,NaN,NaN,NaN,...,2703.22,1843.34,859.88,0.0,24.56,36.48,False,True,False,False
9,4th person,3.0,IL,RENT,30000.0,Not Verified,18.91,NaN,NaN,NaN,...,391.15,322.87,68.28,0.0,53.66,18.91,False,False,False,True


In [64]:
# calculating a risk intensity score because it wont make sense to do analysis on 4 different flags
# the different risk factors will have different weights, but for now lets assume all are equal to not make it too complex
# the flag columns wont be dropped, it may come in handy

In [65]:
risk_columns = [
    'high_debt_burden_flag',
    'severe_credit_event_flag',
    'high_credit_stress_flag',
    'recent_delinquency_flag'
]

df['risk_intensity_score'] = df[risk_columns].mean(axis=1) * 100.0

In [66]:
df

,emp_title,emp_length,state,homeownership,annual_income,verified_income,debt_to_income,annual_income_joint,verification_income_joint,debt_to_income_joint,...,paid_interest,paid_late_fees,cr_util_ratio,correct_paid_total,effective_dti,high_debt_burden_flag,severe_credit_event_flag,high_credit_stress_flag,recent_delinquency_flag,risk_intensity_score
0,Global Config Engineer,3.0,NJ,Mortgage,90000.0,Verified,18.01,NaN,Nan,NaN,...,1015.19,0.0,54.76,True,18.01,False,True,False,False,25.0
1,Warehouse Office Clerk,10.0,HI,Rent,40000.0,Not Verified,5.04,NaN,Nan,NaN,...,150.49,0.0,15.00,True,5.04,False,True,False,False,25.0
2,Assembly,3.0,WI,Rent,40000.0,Source Verified,21.15,NaN,Nan,NaN,...,106.43,0.0,66.13,True,21.15,False,True,False,False,25.0
3,Customer Service,1.0,PA,Rent,30000.0,Not Verified,10.16,NaN,Nan,NaN,...,566.15,0.0,19.67,True,10.16,False,True,False,False,25.0
4,Security Supervisor,10.0,CA,Rent,35000.0,Verified,57.96,57000.0,Verified,37.66,...,754.80,0.0,75.49,True,37.66,False,False,True,False,25.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Owner,10.0,TX,Rent,108000.0,Source Verified,22.28,NaN,Nan,NaN,...,556.14,0.0,39.14,True,22.28,False,True,False,False,25.0
9996,Director,8.0,PA,Mortgage,121000.0,Verified,32.38,NaN,Nan,NaN,...,603.75,0.0,26.59,True,32.38,False,False,False,True,25.0
9997,Toolmaker,10.0,CT,Mortgage,67000.0,Verified,45.26,107000.0,Source Verified,29.57,...,2238.45,0.0,27.55,True,29.57,False,False,False,True,25.0
9998,Manager,1.0,WI,Mortgage,80000.0,Source Verified,11.99,NaN,Nan,NaN,...,391.43,0.0,9.39,True,11.99,False,False,False,False,0.0


## ESTABLISHING CONNECTION STRING

In [67]:
# creating the connection, engine creates the blueprint of the connection
from sqlalchemy import create_engine
engine = create_engine("mssql+pyodbc://@localhost\\SQLEXPRESS/lending club?driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes&TrustServerCertificate=yes")

In [68]:
df.to_sql(name = 'lendings', con=engine, index = False)

1

# SUCCESSFULLY EXPORTED TO MS SQL SERVE